# Demo Template — any park, any variable

Live-demo harness. Three blanks to fill in as you go; everything else is hardcoded so there's minimal code on stage.

**The flow:**

1. 🔍 **fuzzy search** a park name (typos welcome)
2. ✏️ **pick the exact match** from the search output → immediately shows what LOCA2 + WRF data exists for this park
3. ☁️ **start the cluster** (takes ~3-4 min; kick this off early so it's ready by the time you've picked a variable)
4. 🔢 **pick a variable** → fetches time series + spatial snapshots in parallel
5. 📈 **time series plot** — annual trajectory 1950-2100, 4 scenarios, ensemble spread
6. 🗺️ **2×2 spatial plot** — three SSP anomaly maps + Historical reference panel

**Hardcoded (to keep the demo tight):** all 4 scenarios, monthly timescale, historical baseline 1995–2014, future window 2050–2069.

In [ ]:
import os, sys
from time import perf_counter
import coiled
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from andrewAdaptLibrary import (
    ParkCatalog, CatalogExplorer,
    get_climate_data, get_spatial_snapshot, plot_spatial_comparison,
    annual_aggregate, smooth,
    VARIABLE_MAP,
)

NPS_SHP = os.path.join(
    PROJECT_ROOT,
    "USA_National_Park_Service_Lands_20170930_4993375350946852027",
    "USA_Federal_Lands.shp",
)
park_catalog = ParkCatalog(NPS_SHP)
catalog_mon  = CatalogExplorer(timescale="monthly")

SCENARIOS      = ["Historical Climate", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]
HIST_WINDOW    = (1995, 2014)
FUTURE_WINDOW  = (2050, 2069)

COLORS = {
    "Historical Climate": "#333333",
    "SSP 2-4.5": "#4daf4a",
    "SSP 3-7.0": "#ff7f00",
    "SSP 5-8.5": "#e41a1c",
}


# ---- Plot helpers (defined once so the demo cells are one-liners) ----

def plot_time_series(ts_df, variable, park_name):
    """Annual trajectory per scenario with MMEM + 10-90 percentile band."""
    da = (ts_df.set_index(["scenario", "simulation", "time"])[variable]
             .to_xarray().sortby("time"))
    annual = annual_aggregate(da, variable)

    ylabel = {
        "T_Max":  "Annual mean T_Max (°C)",
        "T_Min":  "Annual mean T_Min (°C)",
        "Precip": "Annual total precipitation (mm/yr)",
    }.get(variable, f"Annual {variable}")

    fig, ax = plt.subplots(figsize=(11, 5))
    for scen in SCENARIOS:
        sub = annual.sel(scenario=scen)
        mmem = smooth(sub.mean("simulation"), window=10)
        lo   = smooth(sub.quantile(0.1, dim="simulation"), window=10)
        hi   = smooth(sub.quantile(0.9, dim="simulation"), window=10)
        years = pd.DatetimeIndex(mmem.time.values).year
        ax.plot(years, mmem.values, color=COLORS[scen], linewidth=2, label=scen)
        ax.fill_between(years, lo.values, hi.values, color=COLORS[scen], alpha=0.15)

    # Mark the reference and future windows
    ax.axvspan(*HIST_WINDOW, color="lightgray", alpha=0.35, zorder=0)
    ax.axvspan(*FUTURE_WINDOW, color="lightyellow", alpha=0.5, zorder=0)

    ax.set_xlabel("Year"); ax.set_ylabel(ylabel)
    ax.set_title(f"{park_name} — {variable} annual trajectory, 1950-2100\n"
                 "MMEM (line) + 10-90 percentile (shading), 10-yr smoothed",
                 fontsize=11)
    ax.legend(frameon=False, loc="upper left")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


def plot_2x2(snapshots, boundary, variable, park_name):
    """Three SSP anomaly panels + Historical reference in bottom-right slot."""
    anoms = {f"{s} − Historical": snapshots[s] - snapshots["Historical Climate"]
             for s in SCENARIOS if s != "Historical Climate"}
    cmap_anom = {"T_Max": "RdBu_r", "T_Min": "RdBu_r", "Precip": "BrBG"}.get(variable, "RdBu_r")
    cmap_ref  = {"T_Max": "RdYlBu_r", "T_Min": "RdYlBu_r", "Precip": "YlGnBu"}.get(variable, "viridis")
    units = VARIABLE_MAP.get(variable, {}).get("units_final", "")
    return plot_spatial_comparison(
        anoms, boundary=boundary,
        title=f"{park_name} — {variable} anomaly ({FUTURE_WINDOW[0]}-{FUTURE_WINDOW[1]} vs {HIST_WINDOW[0]}-{HIST_WINDOW[1]}, {units})",
        cmap=cmap_anom,
        reference_panel=(f"Historical ({HIST_WINDOW[0]}-{HIST_WINDOW[1]})",
                         snapshots["Historical Climate"], cmap_ref),
    )

print(park_catalog, "|", catalog_mon)

## 🔍 Box 1: fuzzy search

Type a park name (typos fine — `"yosemit"` still resolves). The result lists the top matches with similarity scores.

In [ ]:
PARK_QUERY = "yosemit"                     # ← BLANK: type a park name

park_catalog.search(PARK_QUERY, top_n=5)

## ✏️ Box 2: pick the exact name

Copy the winning name from Box 1 output. Running this cell also **probes Cal-Adapt live** to show exactly what variables, timescales, and scenarios are available for this park.

In [ ]:
PARK_NAME = "Yosemite National Park"        # ← BLANK: paste exact name from Box 1

boundary = park_catalog.get_boundary(PARK_NAME)
print(f"bounds: {boundary.total_bounds}")
print(f"area:   {boundary.to_crs('EPSG:3310').area.iloc[0] / 1e6:,.0f} km²")
print()
park_catalog.what_is_available(PARK_NAME)

## ☁️ Box 3: start the cluster

Do this *early* — cluster spin-up is ~3-4 min and can overlap with the audience discussion about which variable to pick. 4 workers is plenty for one park × one variable × four scenarios.

In [ ]:
cluster = coiled.Cluster(
    name="demo-template",
    region="us-central1",
    n_workers=4,
    worker_vm_types=["n2-highmem-4"],
    spot_policy="spot_with_fallback",
    idle_timeout="20 minutes",
    package_sync=True,
)
client = cluster.get_client()
client.wait_for_workers(n_workers=3, timeout=180)       # avoid cold-start cancels
print(f"Workers ready: {len(client.scheduler_info()['workers'])}")

## 🔢 Box 4: pick a variable, fetch everything

Pick one of the variables listed in Box 2 — e.g. `T_Max`, `T_Min`, `Precip`.

This cell fires off **two fetches in parallel** on the cluster:

- `get_climate_data` → tidy DataFrame (time × simulation × scenario), collapsed over space inside the park boundary. Feeds the time-series plot.
- `get_spatial_snapshot` → gridded `(lat, lon)` means per scenario, time-averaged over the respective window. Feeds the 2×2 spatial plot.

In [ ]:
VARIABLE = "T_Max"                          # ← BLANK: T_Max, T_Min, or Precip

t0 = perf_counter()
with ThreadPoolExecutor(max_workers=2) as ex:
    ts_future = ex.submit(
        get_climate_data,
        variables=[VARIABLE], scenarios=SCENARIOS,
        boundary=boundary, time_slice=(1950, 2100),
        timescale="monthly", backend="coiled",
        coiled_cluster=cluster, catalog=catalog_mon,
    )
    snap_future = ex.submit(
        get_spatial_snapshot,
        variable=VARIABLE, scenarios=SCENARIOS,
        boundary=boundary, time_period=FUTURE_WINDOW,
        historical_period=HIST_WINDOW,
        coiled_cluster=cluster,
    )
    ts_data   = ts_future.result()[VARIABLE]
    snap_data = snap_future.result()

print(f"fetch wall-clock: {perf_counter()-t0:.0f}s")
print(f"  time series rows: {len(ts_data):,}")
print(f"  spatial snapshots: {list(snap_data.keys())}")
for scen, da in snap_data.items():
    print(f"    {scen:22s} shape={da.shape}, mean={float(da.mean()):.2f}")

## 📈 Box 5: time series plot

In [ ]:
plot_time_series(ts_data, VARIABLE, PARK_NAME);

## 🗺️ Box 6: 2×2 spatial plot

Top row + bottom-left: three SSP anomaly maps (future − historical).
Bottom-right: historical reference panel with its own color scale and colormap.

In [ ]:
plot_2x2(snap_data, boundary, VARIABLE, PARK_NAME);

## Shutdown

In [ ]:
cluster.close()
print("Done.")